# Hierarchical Multi-Agent Pattern: Practical Demo

Our notes' exact example structure:

```
CEO Agent (top-level supervisor)
├─ Research Director   -> Market Research, Competitor Analysis, Academic Literature workers
├─ Analysis Director   -> Financial Analysis, Data Synthesis workers
└─ Production Director -> Writing, Editing workers
```

**The key idea:** a Director is just a supervisor, one level down. So we write **one** reusable function, `run_team()`, that does the supervisor's four jobs (decompose, delegate, collect, synthesise) — and we call it **recursively**: each Director calls it over its own real workers, and the CEO calls it again, one level up, treating the three Directors as if they were its own "workers".

**The real task**, run through both structures below:
*"Produce a short investment brief on NVIDIA's position in the AI chip market: current market and competitor landscape, financial performance, and a polished written summary."*

**The comparison this notebook is built around:** our notes say a single flat supervisor over ~10 workers loses coherence because its context fills with everyone's raw output. We test this directly — same 7 workers, same task, run once **hierarchically** (through the 3 Directors) and once **flat** (one supervisor over all 7 workers directly) — and measure how much raw text actually reaches the final synthesis step in each case.

**What we will do**
1. Setup
2. The real tools and the 7 leaf workers
3. `run_team()`: the one reusable supervisor function
4. The three Directors (each is `run_team()` over its own workers)
5. Hierarchical run: CEO -> 3 Directors -> 7 workers
6. Flat run: 1 supervisor -> all 7 workers directly
7. Comparison and summary

## 1. Setup

In [ ]:
# Bring the api_key from secrets into a variable
from google.colab import userdata

my_api_key = userdata.get('agentic_ai')

In [ ]:
pip install langchain-groq langchain-community ddgs -q # q for silent installation

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = 'openai/gpt-oss-20b',
    temperature = 0, # Creativity level of the model
    api_key = my_api_key,
    max_retries = 6  # retry automatically if Groq's rate limit is reached
)

## 2. The real tools and the 7 leaf workers

### 2.1 Real web search

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun
from ddgs.exceptions import DDGSException

search = DuckDuckGoSearchRun()

def web_search(query):
    try:
        return search.run(query)
    except DDGSException as e:
        return f'(Web search unavailable right now - {e}. No search results to go on.)'

### 2.2 The 7 workers, exactly as named in our notes

Each worker gets a `sub_task` (written by whichever supervisor/director is managing it) and the `context` collected so far **in its own team**. Most workers ignore `context` and just use their sub-task; a few genuinely depend on it.

In [ ]:
def search_worker(sub_task, context, role):
    """Shared logic for the 3 real-world research roles: run a real search, then summarise it."""
    results = web_search(sub_task)[:1500]  # cap length to control token usage through the hierarchy
    prompt = f"""You are the {role}. Summarise what is relevant to the sub-task below, using the search results.

Search results:
{results}

Sub-task: {sub_task}"""
    return llm.invoke(prompt).content

def market_research_worker(sub_task, context):
    return search_worker(sub_task, context, 'market research specialist (AI chip market size and trends)')

def competitor_analysis_worker(sub_task, context):
    return search_worker(sub_task, context, "competitor analysis specialist (NVIDIA's AI chip competitors)")

def academic_literature_worker(sub_task, context):
    return search_worker(sub_task, context, 'academic literature specialist (recent AI accelerator chip research)')

def financial_analysis_worker(sub_task, context):
    return search_worker(sub_task, context, "financial analysis specialist (NVIDIA's recent financial performance)")

In [ ]:
def data_synthesis_worker(sub_task, context):
    # A genuine second-pass job: turn the financial worker's raw findings into clean data points
    financial_findings = context.get('financial_analysis', '(no financial findings available yet)')
    prompt = f"""You are the data synthesis specialist. Extract the key numbers, dates and metrics from
the financial findings below into a short bullet list. Focus on: {sub_task}

Financial findings:
{financial_findings}"""
    return llm.invoke(prompt).content

In [ ]:
def writing_worker(sub_task, context):
    # Uses WHATEVER has been collected so far in its own team - a condensed Director summary
    # in the hierarchical run, or several raw worker outputs in the flat run (see Section 6)
    available = '\n\n'.join(f'{name}: {output}' for name, output in context.items()) or '(no prior findings)'
    prompt = f"""You are the writing specialist. Using the information below, {sub_task}

Available information:
{available}"""
    return llm.invoke(prompt).content

def editing_worker(sub_task, context):
    draft = context.get('writing', '(no draft available)')
    prompt = f"""You are the editing specialist. Polish the draft below for clarity and conciseness.
Focus: {sub_task}

Draft:
{draft}"""
    return llm.invoke(prompt).content

## 3. `run_team()`: the one reusable supervisor function

The same four jobs every time: **decompose** the task into one sub-task per worker, **delegate** to each worker in order (passing along what's been collected so far), **collect** every output, then **synthesise** one summary.

In [ ]:
def decompose(task, worker_names):
    prompt = f"""You are a supervisor managing these workers: {', '.join(worker_names)}.
For each worker, write a short, clear sub-task containing only the part of the task that worker needs.
Order the workers so that any worker needing another's results comes AFTER it.
Decide which worker(s) are needed, in order. Write ONE line per worker in exactly this format, nothing else:
worker_name | sub-task text

Task: {task}"""

    lines = llm.invoke(prompt).content.strip().splitlines()
    plan = []
    for line in lines:
        if '|' in line:
            name, sub_task = line.split('|', 1)
            name = name.strip().lower()
            if name in worker_names:
                plan.append((name, sub_task.strip()))
    return plan

In [ ]:
def run_team(task, workers, team_name, max_context_chars=3000):
    plan = decompose(task, list(workers.keys()))                    # decompose
    print(f"[{team_name}] PLAN:")
    for name, sub_task in plan:
        print(f"  -> {name}: {sub_task}")

    context = {}
    for name, sub_task in plan:                                      # delegate + collect
        context[name] = workers[name](sub_task, context)
        print(f"  [{team_name} -> {name}] {context[name][:150]}...")

    synthesis_input = '\n\n'.join(f'{n}: {o}' for n, o in context.items())

    # Cap what the LLM actually sees, to stay within Groq's rate limit - but keep the
    # untrimmed length below for the comparison in Section 7
    trimmed = synthesis_input[:max_context_chars]
    if len(synthesis_input) > max_context_chars:
        print(f"  [{team_name}] NOTE: context was {len(synthesis_input)} characters, trimmed to {max_context_chars} for synthesis")

    synthesis_prompt = f"""You are the {team_name}. Combine the findings below into one clear summary for: {task}

Findings:
{trimmed}"""
    summary = llm.invoke(synthesis_prompt).content                   # synthesise

    return summary, synthesis_input

## 4. The three Directors

Each Director IS `run_team()`, called over its own real workers. A Director's `context` argument comes from whichever level is managing *it* (the CEO, in Section 5) - that is how a condensed summary from one Director reaches another, with no special-casing.

In [ ]:
research_workers   = {'market_research': market_research_worker,
                       'competitor_analysis': competitor_analysis_worker,
                       'academic_literature': academic_literature_worker}

analysis_workers   = {'financial_analysis': financial_analysis_worker,
                       'data_synthesis': data_synthesis_worker}

production_workers = {'writing': writing_worker, 'editing': editing_worker}

In [ ]:
def research_director(sub_task, context):
    summary, _ = run_team(sub_task, research_workers, 'Research Director')
    return summary

def analysis_director(sub_task, context):
    summary, _ = run_team(sub_task, analysis_workers, 'Analysis Director')
    return summary

In [ ]:
def production_director(sub_task, context):
    # context here is the CEO's context so far: the Research and Analysis Directors' SUMMARIES,
    # not raw worker output - this is the condensed hand-off a hierarchy is meant to provide
    research_summary = context.get('research_director', '')
    analysis_summary = context.get('analysis_director', '')
    task_with_inputs = f"""{sub_task}

Research findings:
{research_summary}

Financial analysis:
{analysis_summary}"""
    summary, _ = run_team(task_with_inputs, production_workers, 'Production Director')
    return summary

## 5. Hierarchical run: CEO -> 3 Directors -> 7 workers

In [ ]:
task = ("Produce a short investment brief on NVIDIA's position in the AI chip market: "
        "current market and competitor landscape, financial performance, and a polished written summary.")

In [ ]:
ceo_team = {'research_director': research_director,
            'analysis_director': analysis_director,
            'production_director': production_director}

print('=' * 70)
print('HIERARCHICAL RUN')
print('=' * 70)
hierarchical_brief, ceo_synthesis_input = run_team(task, ceo_team, 'CEO')

HIERARCHICAL RUN
[CEO] PLAN:
  -> research_director: Gather up-to-date data on NVIDIA's AI chip market position, including market share, competitor landscape, and recent financial performance.
  -> analysis_director: Analyze the collected data to identify key trends, strengths, weaknesses, and financial insights for NVIDIA in the AI chip market.
  -> production_director: Draft a polished investment brief summarizing NVIDIA's AI chip market position, competitor landscape, financial performance, and key investment takeaways.
[Research Director] PLAN:
  -> market_research: Collect the latest market share data for NVIDIA's AI chips, including overall market size, NVIDIA's percentage, and recent financial performance related to AI chip sales.
  -> competitor_analysis: Analyze the competitor landscape for NVIDIA's AI chips, identifying key competitors, their market shares, and recent product releases.
  [Research Director -> market_research] **Key Take‑aways for the “AI‑chip market share” su

In [ ]:
print('FINAL BRIEF (hierarchical):\n')
print(hierarchical_brief)

FINAL BRIEF (hierarchical):

**NVIDIA – Investment Brief (Nov 2026)**  
*Prepared for the Board – CEO’s Summary*

---

### 1. Executive Snapshot  

| Item | Detail |
|------|--------|
| **Market Size** | $30 B AI‑chip revenue (2023) → $70 B by 2027 (≈ 20 % CAGR) |
| **NVIDIA’s Share** | ~70 % of the global AI‑chip market (data‑center dominated) |
| **Key Revenue Driver** | Data‑Center GPUs (A100, H100, Hopper) – $12 B FY 2023, +35 % YoY |
| **Growth Outlook** | FY 2024 guidance: +45 % YoY revenue; FY 2026 projected revenue $28.9 B |
| **Margins** | Gross margin ≈ 70 % (up from 68 % FY 2023); operating margin ≈ 45 % FY 2026 |
| **Cash Position** | $30 B cash & equivalents, $5 B debt – highly leveraged‑free |
| **Investment Thesis** | Buy – 10‑15 % upside over the next 12 months, driven by market dominance, high‑margin business, and a cash‑rich balance sheet. |

---

### 2. Market & Competitive Landscape  

| Segment | 2024 Market Share | Core Offering | Key Differentiator |
|---------|-

## 6. Flat run: 1 supervisor -> all 7 workers directly

Same task, same 7 workers - but no Directors. One supervisor decomposes the task straight into all 7 sub-tasks, and one synthesis call has to combine **everything**: 5 raw worker outputs, a full draft, AND the already-edited version, all at once.

In [ ]:
flat_workers = {**research_workers, **analysis_workers, **production_workers}

print('=' * 70)
print('FLAT RUN')
print('=' * 70)
flat_brief, flat_synthesis_input = run_team(task, flat_workers, 'Supervisor')

FLAT RUN
[Supervisor] PLAN:
  -> market_research: Gather current market size, growth trends, and key drivers for AI chips, focusing on NVIDIA's share.
  -> competitor_analysis: Identify main competitors in AI chip market, compare product offerings, market shares, and recent developments.
  -> academic_literature: Summarize recent academic studies on AI chip performance and market forecasts relevant to NVIDIA.
  -> financial_analysis: Analyze NVIDIA's recent financial statements, focusing on revenue, profit margins, and AI chip segment performance.
  -> data_synthesis: Compile and integrate findings from market_research, competitor_analysis, academic_literature, and financial_analysis into a cohesive dataset.
  -> writing: Draft a concise investment brief summarizing NVIDIA's AI chip market position, competitor landscape, financial performance, and key insights.
  -> editing: Refine the draft for clarity, conciseness, and professional tone, ensuring all facts are accurate and citations 

In [ ]:
print('FINAL BRIEF (flat):\n')
print(flat_brief)

FINAL BRIEF (flat):

**Investment Brief – NVIDIA Corp. (NVDA) – AI‑Chip Market Position**

| Item | Detail |
|------|--------|
| **Ticker** | NVDA |
| **Sector** | Semiconductors – AI & Data‑Center |
| **Investment Thesis** | NVIDIA remains the undisputed leader in the high‑margin GPU‑centric AI‑chip market, with a robust ecosystem, strong data‑center demand, and a clear path to capture the expanding inference‑driven edge/automotive segment. |
| **Target** | Long‑term hold (5‑10 yrs) – upside driven by continued AI‑model complexity, cloud‑first strategy, and the company’s aggressive product roadmap (Hopper, Grace‑CPU, Jetson). |

---

### 1. Market & Competitive Landscape

| Segment | 2023 (est.) | 2028 (proj.) | CAGR 2024‑28 | NVIDIA’s Position |
|---------|-------------|--------------|--------------|-------------------|
| **Global AI‑chip market** | $10–12 B | $25–30 B | ≈ 35 % | 30 % share (≈ $3–4 B) |
| **GPU‑centric AI‑chip sub‑segment** | $6–8 B | $15–18 B | ≈ 35 % | 60 % share (

## 7. Comparison and summary

In [ ]:
print(f"Characters fed to the final synthesis call:")
print(f"  Hierarchical (CEO sees 3 Director SUMMARIES): {len(ceo_synthesis_input)} characters")
print(f"  Flat (1 supervisor sees {len(flat_workers)} RAW worker outputs):       {len(flat_synthesis_input)} characters")

Characters fed to the final synthesis call:
  Hierarchical (CEO sees 3 Director SUMMARIES): 15831 characters
  Flat (1 supervisor sees 7 RAW worker outputs):       27034 characters


**How to read this:** the character count is a concrete, measurable stand-in for "context fills up" from our notes. Read both final briefs above and judge for yourself whether the flat version's synthesis call - combining far more raw text at once - actually reads as less coherent, or whether 7 workers wasn't yet enough to break it (our notes' example uses ~10).

### Summary, from our notes

**Why hierarchy over one flat supervisor:** a single supervisor managing many workers starts to lose coherence as its context fills with everyone's raw output. Hierarchy lets you scale by adding management layers - each Director absorbs its own workers' raw output and hands up only a condensed summary.

**When hierarchy is worth it vs. just added complexity:** worth it when the task has natural domain separations matching different expertise - here, market research, financial analysis and writing genuinely are different skills. A simple "search and summarise" task would not need any of this.

**The cost:** more LLM calls (every Director is its own decompose + synthesise, on top of the CEO's), and more code to maintain - which is exactly why `run_team()` being one reusable function, rather than three separate implementations, matters in practice.